In [1]:
!pip install pyspark delta-spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: importlib_metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0


In [2]:
import pyspark
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

In [3]:
builder = SparkSession.builder \
    .appName("Week7_DeltaLake_Assignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session with Delta Lake Created Successfully!")

Spark Session with Delta Lake Created Successfully!


In [4]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [5]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [6]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [7]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [8]:
print("Total Records :", df.count())
print("Total Columns :", len(df.columns))

Total Records : 9994
Total Columns : 21


In [9]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [10]:
total_rows = df.count()

duplicate_rows = total_rows - df.dropDuplicates().count()

print("Total Rows :", total_rows)
print("Duplicate Rows :", duplicate_rows)

Total Rows : 9994
Duplicate Rows : 0


In [12]:
clean_df = df

In [15]:
clean_df = clean_df.toDF(
    *[
        c.replace(" ", "_").replace("-", "_")
        for c in clean_df.columns
    ]
)

In [16]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("delta_superstore")

In [17]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "delta_superstore"
)

print("Delta Table Created Successfully!")

Delta Table Created Successfully!


In [18]:
delta_table.toDF().show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [19]:
print("Rows in Delta Table :", delta_table.toDF().count())

Rows in Delta Table : 9994


In [21]:
clean_df.select(
    "Row_ID",
    "Order_ID",
    "Customer_Name",
    "Sales",
    "Profit"
).show(10, truncate=False)

+------+--------------+---------------+--------+--------+
|Row_ID|Order_ID      |Customer_Name  |Sales   |Profit  |
+------+--------------+---------------+--------+--------+
|1     |CA-2016-152156|Claire Gute    |261.96  |41.9136 |
|2     |CA-2016-152156|Claire Gute    |731.94  |219.582 |
|3     |CA-2016-138688|Darrin Van Huff|14.62   |6.8714  |
|4     |US-2015-108966|Sean O'Donnell |957.5775|-383.031|
|5     |US-2015-108966|Sean O'Donnell |22.368  |2.5164  |
|6     |CA-2014-115812|Brosina Hoffman|48.86   |14.1694 |
|7     |CA-2014-115812|Brosina Hoffman|7.28    |1.9656  |
|8     |CA-2014-115812|Brosina Hoffman|907.152 |90.7152 |
|9     |CA-2014-115812|Brosina Hoffman|18.504  |5.7825  |
|10    |CA-2014-115812|Brosina Hoffman|114.9   |34.47   |
+------+--------------+---------------+--------+--------+
only showing top 10 rows


In [22]:
from pyspark.sql import Row

incremental_data = [

    # Existing Record (Update)
    Row(
        Row_ID=1,
        Order_ID="CA-2016-152156",
        Order_Date="11/8/2016",
        Ship_Date="11/11/2016",
        Ship_Mode="Second Class",
        Customer_ID="CG-12520",
        Customer_Name="Claire Gute",
        Segment="Consumer",
        Country="United States",
        City="Henderson",
        State="Kentucky",
        Postal_Code=42420,
        Region="East",
        Product_ID="FUR-BO-10001798",
        Category="Furniture",
        Sub_Category="Bookcases",
        Product_Name="Bush Somerset Collection Bookcase",
        Sales=500.00,
        Quantity=2,
        Discount=0.0,
        Profit=120.00
    ),

    # Existing Record (Update)
    Row(
        Row_ID=2,
        Order_ID="CA-2016-152156",
        Order_Date="11/8/2016",
        Ship_Date="11/11/2016",
        Ship_Mode="Second Class",
        Customer_ID="CG-12520",
        Customer_Name="Claire Gute",
        Segment="Consumer",
        Country="United States",
        City="Henderson",
        State="Kentucky",
        Postal_Code=42420,
        Region="East",
        Product_ID="FUR-CH-10000454",
        Category="Furniture",
        Sub_Category="Chairs",
        Product_Name="Hon Deluxe Chair",
        Sales=900.00,
        Quantity=3,
        Discount=0.0,
        Profit=250.00
    ),

    # New Record (Insert)
    Row(
        Row_ID=9995,
        Order_ID="CA-2024-999995",
        Order_Date="01/08/2024",
        Ship_Date="05/08/2024",
        Ship_Mode="Standard Class",
        Customer_ID="HS-50001",
        Customer_Name="Harshit Goel",
        Segment="Consumer",
        Country="United States",
        City="New York",
        State="New York",
        Postal_Code=10001,
        Region="East",
        Product_ID="TEC-100001",
        Category="Technology",
        Sub_Category="Laptops",
        Product_Name="Dell Inspiron Laptop",
        Sales=1500.00,
        Quantity=1,
        Discount=0.0,
        Profit=350.00
    ),

    # New Record (Insert)
    Row(
        Row_ID=9996,
        Order_ID="CA-2024-999996",
        Order_Date="02/08/2024",
        Ship_Date="06/08/2024",
        Ship_Mode="First Class",
        Customer_ID="HS-50002",
        Customer_Name="Aman Sharma",
        Segment="Corporate",
        Country="United States",
        City="Chicago",
        State="Illinois",
        Postal_Code=60601,
        Region="Central",
        Product_ID="OFF-100002",
        Category="Office Supplies",
        Sub_Category="Binders",
        Product_Name="Premium Binder",
        Sales=250.00,
        Quantity=5,
        Discount=0.10,
        Profit=45.00
    )

]

In [23]:
incremental_df = spark.createDataFrame(incremental_data)

In [24]:
incremental_df.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+---------+--------+-----------+-------+---------------+---------------+------------+---------------------------------+------+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment  |Country      |City     |State   |Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                     |Sales |Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+---------+--------+-----------+-------+---------------+---------------+------------+---------------------------------+------+--------+--------+------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  |Consumer |United States|Henderson|Kentucky|42420      |East   |FUR-BO-10001798|Furniture      |Bookcases   |Bush Somerset Collectio

In [25]:
from delta.tables import DeltaTable

delta_table.alias("target").merge(
    incremental_df.alias("source"),
    "target.Row_ID = source.Row_ID"
).whenMatchedUpdate(
    set={
        "Sales": "source.Sales",
        "Profit": "source.Profit",
        "Region": "source.Region"
    }
).whenNotMatchedInsertAll().execute()

print("MERGE Operation Completed Successfully!")

MERGE Operation Completed Successfully!


In [26]:
delta_table.toDF() \
    .filter("Row_ID IN (1,2)") \
    .show(truncate=False)

+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+-----------------------------------------------------------+-----+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode   |Customer_ID|Customer_Name|Segment |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category |Sub_Category|Product_Name                                               |Sales|Quantity|Discount|Profit|
+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+-----------------------------------------------------------+-----+--------+--------+------+
|2     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class|CG-12520   |Claire Gute  |Consumer|United States|Henderson|Kentucky|42420      |East  |FUR-CH-10000454|Furniture|C

In [27]:
delta_table.toDF() \
    .filter("Row_ID >= 9995") \
    .show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+--------+--------+-----------+-------+----------+---------------+------------+--------------------+------+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment  |Country      |City    |State   |Postal_Code|Region |Product_ID|Category       |Sub_Category|Product_Name        |Sales |Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+--------+--------+-----------+-------+----------+---------------+------------+--------------------+------+--------+--------+------+
|9995  |CA-2024-999995|01/08/2024|05/08/2024|Standard Class|HS-50001   |Harshit Goel |Consumer |United States|New York|New York|10001      |East   |TEC-100001|Technology     |Laptops     |Dell Inspiron Laptop|1500.0|1       |0.0     |350.0 |
|9996  |CA-2024-999996|02/08/202

In [28]:
print("Total Rows After MERGE :", delta_table.toDF().count())

Total Rows After MERGE : 9996


In [29]:
before_merge = clean_df.count()
after_merge = delta_table.toDF().count()

print("Rows Before MERGE :", before_merge)
print("Rows After MERGE  :", after_merge)

Rows Before MERGE : 9994
Rows After MERGE  : 9996


In [30]:
delta_table.toDF() \
.select("Row_ID","Sales","Profit","Region") \
.filter("Row_ID IN (1,2)") \
.show(truncate=False)

+------+-----+------+------+
|Row_ID|Sales|Profit|Region|
+------+-----+------+------+
|2     |900.0|250.0 |East  |
|1     |500.0|120.0 |East  |
+------+-----+------+------+



In [31]:
delta_table.toDF() \
.filter("Row_ID IN (9995,9996)") \
.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+--------+--------+-----------+-------+----------+---------------+------------+--------------------+------+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment  |Country      |City    |State   |Postal_Code|Region |Product_ID|Category       |Sub_Category|Product_Name        |Sales |Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+--------+--------+-----------+-------+----------+---------------+------------+--------------------+------+--------+--------+------+
|9995  |CA-2024-999995|01/08/2024|05/08/2024|Standard Class|HS-50001   |Harshit Goel |Consumer |United States|New York|New York|10001      |East   |TEC-100001|Technology     |Laptops     |Dell Inspiron Laptop|1500.0|1       |0.0     |350.0 |
|9996  |CA-2024-999996|02/08/202

In [32]:
delta_table.toDF().show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|   East|FUR-CH-10000454|      Furniture